In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# อินพุตและเอาต์พุตของ Primitives

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** beta release ของโมเดลการประมวลผลใหม่พร้อมให้ใช้งานแล้ว โมเดล directed execution ให้ความยืดหยุ่นมากขึ้นในการปรับแต่ง error mitigation workflow ดูคู่มือ [Directed execution model](/guides/directed-execution-model) สำหรับข้อมูลเพิ่มเติม


<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.4.0
```
</details>
หน้านี้ให้ภาพรวมของอินพุตและเอาต์พุตของ Qiskit Runtime primitives ที่ประมวลผล workloads บน IBM Quantum&reg; compute resources Primitives เหล่านี้ช่วยให้สามารถกำหนด vectorized workloads ได้อย่างมีประสิทธิภาพโดยใช้โครงสร้างข้อมูลที่เรียกว่า **Primitive Unified Bloc (PUB)** PUBs เหล่านี้คือหน่วยงานพื้นฐานที่ QPU ต้องใช้ในการประมวลผล workloads และใช้เป็นอินพุตของเมธอด [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) สำหรับ Sampler และ Estimator primitives ซึ่งประมวลผล workload ที่กำหนดเป็น job จากนั้น หลังจาก job เสร็จสิ้น ผลลัพธ์จะถูกส่งคืนในรูปแบบที่ขึ้นอยู่กับทั้ง PUBs ที่ใช้และ runtime options ที่ระบุจาก Sampler หรือ Estimator primitives
<span id="pubs"></span>
## ภาพรวมของ PUBs
เมื่อเรียกใช้เมธอด [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) ของ primitive argument หลักที่ต้องใช้คือ `list` ของ tuple หนึ่งตัวหรือมากกว่า — หนึ่ง tuple ต่อหนึ่ง circuit ที่รัน แต่ละ tuple นี้ถือเป็น PUB และ elements ที่จำเป็นของแต่ละ tuple ในรายการขึ้นอยู่กับ primitive ที่ใช้ ข้อมูลที่ระบุใน tuples เหล่านี้ยังสามารถจัดเรียงในรูปแบบต่างๆ เพื่อให้มีความยืดหยุ่นใน workload ผ่านการ broadcasting — ซึ่งมีกฎที่อธิบายไว้ใน[ส่วนถัดไป](#broadcasting-rules)

### Estimator PUB
สำหรับ Estimator primitive รูปแบบของ PUB ควรมีค่าไม่เกินสี่ค่า:
- `QuantumCircuit` เดียว ซึ่งอาจมี [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter) objects หนึ่งตัวหรือมากกว่า
- รายการของ observables หนึ่งตัวหรือมากกว่า ซึ่งระบุ expectation values ที่ต้องการประมาณ จัดเรียงเป็น array (เช่น observable เดียวแทนด้วย 0-d array, รายการของ observables เป็น 1-d array และอื่นๆ) ข้อมูลสามารถอยู่ในรูปแบบ `ObservablesArrayLike` ใดก็ได้ เช่น `Pauli`, `SparsePauliOp`, `PauliList` หรือ `str`
   > **Note:** ถ้ามี observables สองตัวที่ commute กันอยู่ใน PUBs ต่างกันแต่มี circuit เดียวกัน จะไม่ถูกประมาณด้วยการวัดเดียวกัน แต่ละ PUB แทน basis สำหรับการวัดที่แตกต่างกัน ดังนั้นจึงต้องมีการวัดแยกกันสำหรับแต่ละ PUB เพื่อให้แน่ใจว่า commuting observables ถูกประมาณด้วยการวัดเดียวกัน ต้องจัดกลุ่มไว้ใน PUB เดียวกัน
- ชุดของค่า parameter เพื่อ bind กับ circuit สามารถระบุเป็น array-like object เดียวที่ index สุดท้ายอยู่บน `Parameter` objects ของ circuit หรือละไว้ (หรือเทียบเท่ากับการตั้งค่าเป็น `None`) ถ้า circuit ไม่มี `Parameter` objects
- (ไม่บังคับ) ความแม่นยำเป้าหมายสำหรับ expectation values ที่ต้องการประมาณ

### Sampler PUB
สำหรับ Sampler primitive รูปแบบของ PUB tuple มีค่าไม่เกินสามค่า:
- `QuantumCircuit` เดียว ซึ่งอาจมี [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter) objects หนึ่งตัวหรือมากกว่า
   *หมายเหตุ: Circuit เหล่านี้ควรมีคำสั่งการวัดสำหรับแต่ละ qubit ที่ต้องการ sample ด้วย*
- ชุดของค่า parameter เพื่อ bind กับ circuit $\theta_k$ (จำเป็นเฉพาะเมื่อมี `Parameter` objects ที่ต้อง bind ตอน runtime)
- (ไม่บังคับ) จำนวน shots สำหรับวัด circuit
---

โค้ดต่อไปนี้แสดงตัวอย่างชุดของ vectorized inputs ให้กับ `Estimator` primitive และรันบน IBM&reg; backend เป็น `RuntimeJobV2 ` object เดียว

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray
from qiskit.primitives import StatevectorEstimator


import numpy as np

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit without providing a backend
pm = generate_preset_pass_manager(optimization_level=1)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 10),
        np.linspace(-4 * np.pi, 4 * np.pi, 10),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator = StatevectorEstimator()
estimator_pub = (transpiled_circuit, observables, params)

# Run the transpiled circuit
# using the set of parameters and observables.

job = estimator.run([estimator_pub])
result = job.result()

<span id="broadcasting"></span>
### Broadcasting rules

The PUBs aggregate elements from multiple arrays (observables and parameter values) by following the same broadcasting rules as NumPy. This section briefly summarizes those rules.  For a detailed explanation, see the [NumPy broadcasting rules documentation](https://numpy.org/doc/stable/user/basics.broadcasting.html).

Rules:

* Input arrays do not need to have the same number of dimensions.
  * The resulting array will have the same number of dimensions as the input array with the largest dimension.
  * The size of each dimension is the largest size of the corresponding dimension.
  * Missing dimensions are assumed to have size one.
* Shape comparisons start with the rightmost dimension and continue to the left.
* Two dimensions are compatible if their sizes are equal or if one of them is 1.

Examples of array pairs that broadcast:

```text
A1     (1d array):      1
A2     (2d array):  3 x 5
Result (2d array):  3 x 5


A1     (3d array):  11 x 2 x 7
A2     (3d array):  11 x 1 x 7
Result (3d array):  11 x 2 x 7
```

Examples of array pairs that do not broadcast:

```text
A1     (1d array):  5
A2     (1d array):  3

A1     (2d array):      2 x 1
A2     (3d array):  6 x 5 x 4 # This would work if the middle dimension were 2, but it is 5.
```

`Estimator` returns one expectation value estimate for each element of the broadcasted shape.

Here are some examples of common patterns expressed in terms of array broadcasting.  Their accompanying visual representation is shown in the figure that follows:


Parameter value sets are represented by n x m arrays, and observable arrays are represented by one or more single-column arrays. For each example in the previous code, the parameter value sets are combined with their observable array to create the resulting expectation value estimates.

 - *Example 1*: (broadcast single observable) has a parameter value set that is a 5x1 array and a 1x1 observables array.  The one item in the observables array is combined with each item in the parameter value set to create a single 5x1 array where each item is a combination of the original item in the parameter value set with the item in the observables array.

 - *Example 2*: (zip) has a 5x1 parameter value set and a 5x1 observables array.  The output is a 5x1 array where each item is a combination of the nth item in the parameter value set with the nth item in the observables array.

  - *Example 3*: (outer/product) has a 1x6 parameter value set and a 4x1 observables array.  Their combination results in a 4x6 array that is created by combining each item in the parameter value set with *every* item in the observables array, and thus each parameter value becomes an entire column in the output.

  - *Example 4*: (Standard nd generalization) has a 3x6 parameter value set array and two 3x1 observables array.  These combine to create two 3x6 output arrays in a similar manner to the previous example.

![This image illustrates several visual representations of array broadcasting.](../docs/images/guides/primitive-input-output/broadcasting.svg "Visual representation of broadcasting")

In [2]:
# Broadcast single observable
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = SparsePauliOp("ZZZ")  # shape ()
# >> pub result has shape (5,)

# Zip
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = [
    SparsePauliOp(pauli) for pauli in ["III", "XXX", "YYY", "ZZZ", "XYZ"]
]  # shape (5,)
# >> pub result has shape (5,)

# Outer/Product
parameter_values = np.random.uniform(size=(1, 6))  # shape (1, 6)
observables = [
    [SparsePauliOp(pauli)] for pauli in ["III", "XXX", "YYY", "ZZZ"]
]  # shape (4, 1)
# >> pub result has shape (4, 6)

# Standard nd generalization
parameter_values = np.random.uniform(size=(3, 6))  # shape (3, 6)
observables = [
    [
        [SparsePauliOp(["XII"])],
        [SparsePauliOp(["IXI"])],
        [SparsePauliOp(["IIX"])],
    ],
    [
        [SparsePauliOp(["ZII"])],
        [SparsePauliOp(["IZI"])],
        [SparsePauliOp(["IIZ"])],
    ],
]  # shape (2, 3, 1)
# >> pub result has shape (2, 3, 6)

### กฎการ Broadcasting
PUBs รวม elements จาก arrays หลายตัว (observables และค่า parameter) โดยทำตามกฎการ broadcasting เดียวกับ NumPy ส่วนนี้สรุปกฎเหล่านั้นโดยย่อ สำหรับคำอธิบายโดยละเอียด ดู[เอกสาร NumPy broadcasting rules](https://numpy.org/doc/stable/user/basics.broadcasting.html)

กฎ:

* Input arrays ไม่จำเป็นต้องมีจำนวน dimensions เท่ากัน
  * Array ผลลัพธ์จะมีจำนวน dimensions เท่ากับ input array ที่มี dimension ใหญ่ที่สุด
  * ขนาดของแต่ละ dimension คือขนาดที่ใหญ่ที่สุดของ dimension ที่ตรงกัน
  * Dimensions ที่หายไปถือว่ามีขนาดหนึ่ง
* การเปรียบเทียบ shape เริ่มจาก dimension ขวาสุดและดำเนินต่อไปทางซ้าย
* Dimensions สองตัวเข้ากันได้ถ้าขนาดเท่ากันหรือถ้าตัวหนึ่งเป็น 1

ตัวอย่างคู่ของ arrays ที่ broadcast ได้:

In [3]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10)), metadata={'target_precision': 0.0, 'circuit_metadata': {}})], metadata={'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10))

And this DataBin has attributes: dict_keys(['evs', 'stds'])
Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). 

The expectation values measured from this PUB are: 
[[ 3.06161700e-16  4.52395120e-01  4.36594428e-01  2.16506351e-01
   6.33718361e-01 -6.33718361e-01 -2.16506351e-01 -4.36594428e-01
  -4.52395120e-01 -3.06161700e-16]
 [ 1.22464680

ตัวอย่างคู่ของ arrays ที่ broadcast ไม่ได้:

In [4]:
from qiskit.primitives import StatevectorSampler

# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

# transpile the circuit
transpiled_circuit = pm.run(circuit)

sampler = StatevectorSampler()

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data
print(f"Databin: {data}\n")

# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=10>))

BitArray: BitArray(<shape=(), num_shots=1024, num_bits=10>)

The shape of register `meas` is (1024, 2).

The bytes in register `alpha`, shot by shot:
[[  0   0]
 [  3 255]
 [  0   0]
 ...
 [  3 255]
 [  3 255]
 [  3 255]]



`EstimatorV2` คืน expectation value estimate หนึ่งค่าสำหรับแต่ละ element ของ shape ที่ broadcast แล้ว

นี่คือตัวอย่างของ patterns ทั่วไปที่แสดงในรูปของ array broadcasting พร้อมกับการแสดงภาพแบบ visual ในรูปถัดไป:

ชุดค่า parameter แทนด้วย n x m arrays และ observable arrays แทนด้วย single-column arrays หนึ่งตัวหรือมากกว่า สำหรับแต่ละตัวอย่างในโค้ดก่อนหน้า ชุดค่า parameter ถูกรวมกับ observable array เพื่อสร้าง expectation value estimates ที่ได้

 - *ตัวอย่างที่ 1*: (broadcast single observable) มีชุดค่า parameter เป็น array 5x1 และ observables array แบบ 1x1 item เดียวใน observables array ถูกรวมกับแต่ละ item ในชุดค่า parameter เพื่อสร้าง array 5x1 เดียวที่แต่ละ item เป็นการรวมกันของ item ต้นฉบับในชุดค่า parameter กับ item ใน observables array

 - *ตัวอย่างที่ 2*: (zip) มีชุดค่า parameter 5x1 และ observables array 5x1 ผลลัพธ์คือ array 5x1 ที่แต่ละ item เป็นการรวมกันของ item ที่ n ในชุดค่า parameter กับ item ที่ n ในชุด observables array

  - *ตัวอย่างที่ 3*: (outer/product) มีชุดค่า parameter 1x6 และ observables array 4x1 การรวมกันของพวกมันให้ผล array 4x6 ที่สร้างโดยการรวมแต่ละ item ในชุดค่า parameter กับ *ทุก* item ใน observables array ดังนั้นแต่ละค่า parameter จึงกลายเป็นคอลัมน์ทั้งคอลัมน์ในผลลัพธ์

  - *ตัวอย่างที่ 4*: (Standard nd generalization) มีชุดค่า parameter array 3x6 และ observables array สองตัวแบบ 3x1 พวกมันรวมกันเพื่อสร้าง output arrays 3x6 สองตัวในลักษณะเดียวกับตัวอย่างก่อนหน้า

![ภาพนี้แสดง visual representations หลายแบบของ array broadcasting](../docs/images/guides/primitive-input-output/broadcasting.svg "Visual representation of broadcasting")

In [5]:
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'0000000000': 492, '1111111111': 532}


<Admonition type="tip" title="SparsePauliOp">
แต่ละ `SparsePauliOp` นับเป็น element เดียวในบริบทนี้ โดยไม่คำนึงถึงจำนวน Paulis ที่มีอยู่ใน `SparsePauliOp` ดังนั้น สำหรับจุดประสงค์ของกฎ broadcasting เหล่านี้ elements ต่อไปนี้ทั้งหมดมี shape เดียวกัน:

In [6]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=1024, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=1024, num_bits=9>)


รายการของ operators ต่อไปนี้ แม้จะเทียบเท่ากันในแง่ของข้อมูลที่มีอยู่ มี shape ต่างกัน:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object,
        |       |     ## which includes data such as the following:
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object,
        |       |     ## which includes data such as the following:
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```
</Admonition>
---
title: "อินพุตและเอาต์พุตของ Primitives (ต่อ)"
sidebar_label: "อินพุตและเอาต์พุตของ Primitives (ต่อ)"
description: "ทำความเข้าใจรูปแบบอินพุตและเอาต์พุต (รวมถึง Primitive Unified Blocs หรือ PUBs) ของ primitives — ส่วนที่ 2"
notebook_path: "guides/primitive-input-output.ipynb"
---



## ภาพรวมของ primitive outputs
เมื่อส่ง PUBs หนึ่งตัวหรือมากกว่าไปยัง QPU เพื่อประมวลผลและ job เสร็จสิ้นสำเร็จ ข้อมูลจะถูกส่งคืนในรูปแบบ [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) container object ที่เข้าถึงได้โดยการเรียก `RuntimeJobV2.result()` method `PrimitiveResult` มี iterable list ของ [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) objects ที่มีผลลัพธ์การประมวลผลสำหรับแต่ละ PUB ขึ้นอยู่กับ primitive ที่ใช้ ข้อมูลเหล่านี้จะเป็นทั้ง expectation values และ error bars ในกรณีของ Estimator หรือ samples ของ circuit output ในกรณีของ Sampler

แต่ละ element ของ list นี้ตรงกับแต่ละ PUB ที่ส่งให้กับเมธอด `run()` ของ primitive (เช่น job ที่ส่งพร้อม 20 PUBs จะคืน `PrimitiveResult` object ที่มีรายการ 20 `PubResult` หนึ่งตัวตรงกับแต่ละ PUB)

แต่ละ `PubResult` objects เหล่านี้มีทั้ง `data` และ `metadata` attribute `data` attribute คือ [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) แบบกำหนดเองที่มีค่าการวัดจริง ค่าเบี่ยงเบนมาตรฐาน และอื่นๆ `DataBin` นี้มี attributes ต่างกันขึ้นอยู่กับ shape หรือโครงสร้างของ PUB ที่เกี่ยวข้องและ error mitigation options ที่ระบุโดย primitive ที่ใช้ส่ง job (เช่น [ZNE](./error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) หรือ [PEC](./error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)) ในขณะที่ `metadata` attribute มีข้อมูลเกี่ยวกับ runtime และ error mitigation options ที่ใช้ (อธิบายในส่วน[ข้อมูล metadata ของผลลัพธ์](#result-metadata) ในหน้านี้)

นี่คือโครงร่างภาพของโครงสร้างข้อมูล `PrimitiveResult`:

<Tabs>
    <TabItem value="estimator" label="Estimator Output">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── NAME_OF_CLASSICAL_REGISTER
        │       │   └── BitArray of count data for first PUB (default is 'meas')
        |       |
        │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
        │           └── BitArray of count data (exists only if more than one
        |                 ClassicalRegister was specified in the circuit)
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       └── NAME_OF_CLASSICAL_REGISTER
        |           └── BitArray of count data for second PUB
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
    <TabItem value="sampler" label="Sampler Output">

In [7]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == "0b1"
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

# concatenate the bitstrings in `alpha` and `beta` to "merge" the results of the two
# registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")

The shape of register `alpha` is (1024, 1).
The bytes in register `alpha`, shot by shot:
[[1]
 [1]
 [1]
 ...
 [0]
 [0]
 [1]]

The shape of register `beta` is (1024, 2).
The bytes in register `beta`, shot by shot:
[[  1 255]
 [  1 255]
 [  1 255]
 ...
 [  0   0]
 [  0   0]
 [  1 255]]

The shape of `beta` after post-selection is (0, 2).
The bytes in `beta` after post-selection:
[]
The shape of `beta` after bit-wise slicing is (1024, 1).
The bytes in `beta` after bit-wise slicing:
[[7]
 [7]
 [7]
 ...
 [0]
 [0]
 [7]]

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  1 255]
 [  1 255]
 [  1 255]
 [  1 255]
 [  1 255]]



Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: -0.017578125
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: -0.017578125

The shape of the merged results is (1024, 2).
The bytes of the merged results:
[[  3 255]
 [  3 255]
 [  3 255]
 ...
 [  0   0]
 [  0   0]
 [  3 255]]



</TabItem>
</Tabs>

พูดง่ายๆ คือ job เดียวคืน `PrimitiveResult` object และมีรายการของ `PubResult` objects หนึ่งตัวหรือมากกว่า `PubResult` objects เหล่านี้จัดเก็บข้อมูลการวัดสำหรับแต่ละ PUB ที่ส่งให้กับ job

แต่ละ `PubResult` มีรูปแบบและ attributes ต่างกันตามประเภทของ primitive ที่ใช้สำหรับ job รายละเอียดอธิบายไว้ด้านล่าง

### Estimator output
แต่ละ `PubResult` สำหรับ Estimator primitive มีอย่างน้อย array ของ expectation values (`PubResult.data.evs`) และค่าเบี่ยงเบนมาตรฐานที่เกี่ยวข้อง (ทั้ง `PubResult.data.stds` หรือ `PubResult.data.ensemble_standard_error` ขึ้นอยู่กับ `resilience_level` ที่ใช้) แต่สามารถมีข้อมูลเพิ่มเติมขึ้นอยู่กับ error mitigation options ที่ระบุ

โค้ดด้านล่างอธิบายรูปแบบ `PrimitiveResult` (และ `PubResult` ที่เกี่ยวข้อง) สำหรับ job ที่สร้างด้านบน

In [8]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'version' : 2,

The metadata of the PubResult result is:
'shots' : 1024,
'circuit_metadata' : {},


## Next steps

<Admonition type="tip" title="Recommendations">

  -  Review the [Qiskit primitives](/docs/api/qiskit/primitives) API.
  -  Review the [Qiskit Aer primitives](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) API.
  -  Learn more about the [Qiskit Runtime primitives](/docs/guides/qiskit-runtime-primitives).
  -  Review the [Qiskit Runtime Estimator](/docs/api/qiskit-ibm-runtime/estimator-v2) API.
  -  Review the [Qiskit Runtime Sampler](/docs/api/qiskit-ibm-runtime/sampler-v2) API.
</Admonition>